Notebook examines parameter recapitulation using synthetic data.

In [1]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import scMPRAforge as scm

2025-07-28 11:36:42.456523: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-28 11:36:42.461162: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-07-28 11:36:42.461184: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
#autoreload for dev
%load_ext autoreload
%autoreload 2

In [3]:
#load ground truth
ground_truth_mu=pd.read_csv("../../notebooks/demos/parameter_extraction_demo/synthetic_ground_truth_carp.tsv",sep="\t")
ground_truth_mu=ground_truth_mu.set_index(['cell_type','cre_id'])
ground_truth_mu.head()

true_mu
cell_type cre_id             
brain     reference         1
          mediumbody       10
          everybody       114
          redgene          30
          neurogene        99

In [4]:
#start a lightweighrt cluster
from dask.distributed import Client, LocalCluster
cluster=LocalCluster()
client = Client(cluster)

# Parameter extraction

In [5]:
dat=scm.scMPRA_data.from_tsv("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres_v3.tsv")
dat.ortho_filter()
dat.set_negative_controls(["nobody","weak"])
dat.set_reference_cell("liver")

primordial=scm.ortho()
primordial.criss_cross(client=client,dat=dat)
primordial.extract_params(client)

scMPRAforge: INFO: Dropped 0 of 27 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [12]:
scm.versus_truth(ground_truth_mu,primordial)

,by,mean_squared_error,mean_biased_error,mean_absolute_percent_error
0,cell_type,2.893974,-0.375634,11.840392
1,cre_id,3.112163,-0.336582,11.455252


In [ ]:
cluster.close()